<a href="https://colab.research.google.com/github/kuds/rl-drone/blob/main/notebooks/%5BMultiple%20Targets%5D%20Soft%20Actor-Critic%20(SAC).ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# [Multiple Targets] Soft Actor-Critic (SAC)

Train a Soft Actor-Critic agent on `DroneHoverEnv` with both
`randomize_target=True` **and** `move_on_contact=True`. After every successful
contact with the goal sphere, the env teleports the target to a new random
location and the agent has to chase it again. A single episode therefore
becomes a chain of mini hover tasks, and the policy has to learn to
*re-acquire* a moving target rather than just settle into a single hover.

| Field | Value |
| --- | --- |
| Environment | `DroneHoverEnv(move_on_contact=True, randomize_target=True)` |
| Algorithm | Soft Actor-Critic (SAC, off-policy, continuous control) |
| Policy | `MlpPolicy` from Stable-Baselines3 |
| Default total steps | 5,000,000 |
| Episode length | 2,500 steps |
| Vectorized envs | 4 |
| Sphere size | 0.10 m |
| Observation / reward normalization | `VecNormalize` |

## What this notebook does

1. Installs `rl-drone`, configures EGL for MuJoCo, and verifies GPU access.
2. Builds the artifact directory tree for the `MultipleTargets` run.
3. Trains SAC on the multi-target hover task with the full callback stack
   (eval, checkpoint, video record, plots, config snapshot, `VecNormalize`
   save).
4. Loads the best model, evaluates it, and records a longer rollout video so
   you can watch it chain several target captures together.


## 1. Environment Setup & Dependencies

Configure the NVIDIA EGL ICD so MuJoCo can render off-screen on the Colab
GPU, install the `rl-drone` package directly from GitHub (which pulls in
`mujoco`, `gymnasium`, and `stable-baselines3`), and verify that MuJoCo can
compile a trivial XML model. Finally, install `ffmpeg` + `mediapy` for video
recording and clear the install spam from the output area.


In [ ]:
# Set up GPU rendering.
import distutils.util
import os
import subprocess
if subprocess.run('nvidia-smi').returncode:
  raise RuntimeError(
      'Cannot communicate with GPU. '
      'Make sure you are using a GPU Colab runtime. '
      'Go to the Runtime menu and select Choose runtime type.')

# Add an ICD config so that glvnd can pick up the Nvidia EGL driver.
# This is usually installed as part of an Nvidia driver package, but the Colab
# kernel doesn't install its driver via APT, and as a result the ICD is missing.
# (https://github.com/NVIDIA/libglvnd/blob/master/src/EGL/icd_enumeration.md)
NVIDIA_ICD_CONFIG_PATH = '/usr/share/glvnd/egl_vendor.d/10_nvidia.json'
if not os.path.exists(NVIDIA_ICD_CONFIG_PATH):
  with open(NVIDIA_ICD_CONFIG_PATH, 'w') as f:
    f.write("""{\n
    "file_format_version" : "1.0.0",\n
    "ICD" : {\n
        "library_path" : "libEGL_nvidia.so.0"\n
    }\n
}\n
""")

# Configure MuJoCo to use the EGL rendering backend (requires GPU)
print('Setting environment variable to use GPU rendering:')
%env MUJOCO_GL=egl

!pip install -q "rl-drone[train] @ git+https://github.com/kuds/rl-drone.git"

# Check if installation was successful.
try:
  print('Checking that the installation succeeded:')
  import mujoco
  mujoco.MjModel.from_xml_string('<mujoco/>')
except Exception as e:
  raise e from RuntimeError(
      'Something went wrong during installation. Check the shell output above '
      'for more information.\n'
      'If using a hosted Colab runtime, make sure you enable GPU acceleration '
      'by going to the Runtime menu and selecting "Choose runtime type".')

print('Installation successful.')

# Graphics and plotting.
print('Installing mediapy:')
!command -v ffmpeg >/dev/null || (apt update && apt install -y ffmpeg)
!pip install -q mediapy

from IPython.display import clear_output
clear_output()

## 2. Imports

Bring in the standard library, NumPy / Torch, MuJoCo + Gymnasium, the
Stable-Baselines3 SAC pieces, and the helpers from the `rl_drone` package:
the environment, the run-paths builder, the plotting and summary utilities,
and the custom training callbacks.


In [ ]:
# --- Standard Library ---
import os
import platform
import shutil
from datetime import datetime
from importlib.metadata import version

# --- Third-Party ---
import numpy as np
import torch
import mediapy as media

# --- Stable Baselines3 ---
from stable_baselines3 import SAC
from stable_baselines3.common.env_checker import check_env
from stable_baselines3.common.env_util import make_vec_env
from stable_baselines3.common.evaluation import evaluate_policy
from stable_baselines3.common.callbacks import CallbackList, CheckpointCallback, EvalCallback
from stable_baselines3.common.vec_env import VecNormalize, VecVideoRecorder

# --- rl-drone package ---
from rl_drone.envs.drone_hover import DroneHoverEnv
from rl_drone.utils.model_xml import setup_mujoco_model
from rl_drone.callbacks import (
    VideoRecordCallback,
    TrainingPlotsCallback,
    ConfigSaveCallback,
    VecNormalizeSaveCallback,
    ReformatEvalCallback,
)
from rl_drone.utils.plotting import plot_learning_curves
from rl_drone.utils.paths import build_run_paths
from rl_drone.utils.summary import write_stage_summary

# --- Environment Specific (Colab) ---
try:
    import google.colab.drive
except ImportError:
    pass  # Gracefully handle if not running in Colab

np.set_printoptions(precision=3, suppress=True, linewidth=100)

## 3. Verify Library Versions

Print the versions of every key dependency (Python, Torch, CUDA, Gymnasium,
NumPy, MuJoCo, Stable-Baselines3, and `rl-drone`) so the run is reproducible
and any future regressions are easy to diagnose.


In [ ]:
print(f"Python Version: {platform.python_version()}")
print(f"Torch Version: {version('torch')}")
print(f"Is Cuda Available: {torch.cuda.is_available()}")
print(f"Cuda Version: {torch.version.cuda}")
print(f"Gymnasium Version: {version('gymnasium')}")
print(f"Robot Description Version: {version('robot_descriptions')}")
print(f"Numpy Version: {version('numpy')}")
print(f"Mujoco Version: {version('mujoco')}")
print(f"Stable-Baselines3 Version: {version('stable-baselines3')}")
print(f"Matplotlib Version: {version('matplotlib')}")
print(f"rl-drone Version: {version('rl-drone')}")

## 4. Run Configuration & Artifact Layout

Pick the algorithm (`SAC`), the environment string, and the training
hyperparameters, then call `build_run_paths` to produce a unique timestamped
directory tree (Google Drive by default) for every artifact this run will
create. All later cells use the returned `paths` object so models, plots,
checkpoints, videos, and TensorBoard logs end up neatly grouped under one
folder per run.


In [ ]:
rl_type = "SAC"
project_name = "BitCrazy"
env_str = "MultipleTargets"
name_prefix = f"{env_str}_{rl_type}".lower()

use_google_drive = True
parent_path = None
if use_google_drive:
    parent_path = "/content/gdrive"
    google.colab.drive.mount(parent_path, force_remount=True)

paths = build_run_paths(
    project_name=project_name,
    env_str=env_str,
    rl_type=rl_type,
    parent_path=parent_path,
    use_google_drive=use_google_drive,
)

# Backwards-compatible aliases used throughout the notebook
log_dir = paths.base_dir
model_folder_path = paths.run_dir

hyperparams = {
    "project_name": project_name,
    "env_str": env_str,
    "rl_type": rl_type,
    "eval_freq": 25_000,
    "n_envs": 4,
    "total_timesteps": 5_000_000,
    "log_dir": log_dir,
    "episode_length": 2_500,
    "sphere_size": 0.1,
}

In [ ]:
# Create artifact directories for this run
paths.makedirs()

## 5. Configure the MuJoCo Drone Model

Edit the bundled MuJoCo XML model in place to apply the chosen target sphere
size. This must run **before** any environment is constructed because the
environment constructor reads the model from disk.


In [ ]:
setup_mujoco_model(sphere_size=hyperparams["sphere_size"])

## 6. Inspect the Environment

Quick smoke test: instantiate `DroneHoverEnv(move_on_contact=True,
randomize_target=True)` once, print its observation and action spaces, then
tear it back down. This catches schema mismatches before kicking off a long
training run.


In [ ]:
env = DroneHoverEnv(move_on_contact=True, randomize_target=True)
print("Observation Space Size: ", env.observation_space.shape)
print("Actions Space: ", env.action_space)
env.close()

## 7. Vectorized Environment Factory

Define the `make_env` factory used by `make_vec_env` to spin up parallel
training and evaluation workers. Every constructed env has both
`move_on_contact=True` and `randomize_target=True`, so the goal sphere
re-spawns at a new random location every time the drone makes contact with
it. The factory also runs `check_env` so any Gym API violations fail fast.


In [ ]:
def make_env():
    env = DroneHoverEnv(
        sphere_size=hyperparams["sphere_size"],
        episode_len=hyperparams["episode_length"],
        move_on_contact=True,
        randomize_target=True,
        render_mode="rgb_array",
    )
    check_env(env)
    return env

## 8. Train the SAC Agent

Build the training and evaluation `VecNormalize` wrappers, register the full
callback stack — checkpoints, periodic evaluation, CSV reformatting, video
capture, learning-curve plots, and a hyperparameter snapshot — and call
`model.learn()`. After training, the script saves the final model along
with the `VecNormalize` statistics for both train and eval envs so they can
be reloaded later.

The callbacks are deliberately verbose so that even a stopped run still
leaves useful artifacts on disk.


In [ ]:
# Create Training environment
env = make_vec_env(make_env,
                   n_envs=hyperparams["n_envs"],
                   monitor_dir=paths.monitor_dir)

normalized_env = VecNormalize(env,
                              training=True,
                              norm_obs=True,
                              norm_reward=True)

# Create Evaluation environment
env_val = make_vec_env(make_env, n_envs=1)
normalized_env_val = VecNormalize(env_val,
                                  training=False,
                                  norm_obs=True,
                                  norm_reward=True)

# Callbacks
checkpoint_callback = CheckpointCallback(
    save_freq=hyperparams["eval_freq"],
    save_path=paths.checkpoints_dir,
    name_prefix="rl_model",
    save_replay_buffer=False,
    save_vecnormalize=True,
)

vec_normalize_save_callback = VecNormalizeSaveCallback(
    save_path=paths.run_dir,
)

eval_cb = EvalCallback(
    normalized_env_val,
    best_model_save_path=paths.run_dir,
    log_path=paths.run_dir,
    render=False,
    deterministic=True,
    n_eval_episodes=20,
    eval_freq=hyperparams["eval_freq"],
    callback_on_new_best=vec_normalize_save_callback,
)

reformat_eval_callback = ReformatEvalCallback(
    save_path=paths.run_dir,
    eval_file=os.path.join(paths.run_dir, "evaluations.npz"),
    save_freq=hyperparams["eval_freq"],
    csv_file_name=name_prefix,
)

video_record_callback = VideoRecordCallback(
    make_env_fn=make_env,
    save_path=paths.videos_dir,
    video_length=10_000,
    save_freq=hyperparams["eval_freq"],
    name_prefix=name_prefix,
)

training_plots_callback = TrainingPlotsCallback(
    eval_file=os.path.join(paths.run_dir, "evaluations.npz"),
    save_path=paths.plots_dir,
    save_freq=hyperparams["eval_freq"],
    name_prefix=name_prefix,
)

config_save_callback = ConfigSaveCallback(
    save_path=paths.run_dir,
    hyperparams=hyperparams,
    run_name=f"{rl_type}_{env_str}",
)

callback_list = CallbackList([
    checkpoint_callback,
    eval_cb,
    reformat_eval_callback,
    video_record_callback,
    training_plots_callback,
    config_save_callback,
])

# Train
model = SAC("MlpPolicy",
            normalized_env,
            verbose=0,
            tensorboard_log=paths.tensorboard_dir)

training_start_time = datetime.now()

model.learn(total_timesteps=hyperparams["total_timesteps"],
            callback=callback_list,
            progress_bar=False)
training_end_time = datetime.now()

# Save the model
model.save(os.path.join(paths.run_dir, "final_model"))

mean_reward, std_reward = evaluate_policy(model,
                                          normalized_env_val,
                                          deterministic=True,
                                          n_eval_episodes=20)
print(f"Final Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")

normalized_env.save(os.path.join(paths.run_dir, "final_normalized_env.pkl"))
normalized_env_val.save(os.path.join(paths.run_dir, "final_normalized_env_val.pkl"))

env.close()
env_val.close()
normalized_env.close()
normalized_env_val.close()

## 9. Evaluate the Best Model & Record Video

Reload `best_model.zip` together with its matching `VecNormalize`
statistics, run a deterministic evaluation, write a human-readable
stage-summary report, and record a long MP4 rollout so you can watch the
trained policy chain several target captures together. The cell also
accumulates and prints the un-normalized total episode reward.


In [ ]:
# Create Evaluation environment
env_val = make_vec_env(make_env, n_envs=1)
normalized_env_val = VecNormalize.load(
    os.path.join(paths.run_dir, "best_model_vec_normalize.pkl"),
    venv=env_val,
)

# Load the best model
best_model_path = os.path.join(paths.run_dir, "best_model")
best_model = SAC.load(best_model_path, env=normalized_env_val)

mean_reward, std_reward = evaluate_policy(best_model,
                                          normalized_env_val,
                                          deterministic=True,
                                          n_eval_episodes=20)
print(f"Best Model - Mean reward: {mean_reward:.2f} +/- {std_reward:.2f}")


# Write a human-readable stage summary report.
write_stage_summary(
    save_dir=paths.run_dir,
    model=best_model,
    eval_env=normalized_env_val,
    project_name=project_name,
    env_str=env_str,
    algorithm=rl_type,
    total_timesteps=hyperparams["total_timesteps"],
    training_start=training_start_time,
    training_end=training_end_time,
    description="SAC training run for drone multi-target task.",
    eval_file=os.path.join(paths.run_dir, "evaluations.npz"),
    n_eval_episodes=30,
)

# Record video of the best model
video_length = 10_000
best_model_file_name = f"best_model_{name_prefix}"
env = VecVideoRecorder(
    normalized_env_val,
    paths.videos_dir,
    video_length=video_length,
    record_video_trigger=lambda x: x == 0,
    name_prefix=best_model_file_name,
)

obs = env.reset()
total_reward = 0
for _ in range(video_length):
    action, _states = best_model.predict(obs, deterministic=True)
    obs, rewards, done, info = env.step(action)
    total_reward += env.get_original_reward()
    env.render()
    if done:
        break

env.close()
normalized_env_val.close()
print(f"Total reward: {total_reward[0]:.2f}")

## 10. Plot Learning Curves

Render the reward / episode-length learning curves from the
`evaluations.npz` file produced by `EvalCallback`. The same curves are also
auto-saved during training by `TrainingPlotsCallback` — this final plot is
just a convenient summary at the end of the run.


In [ ]:
plot_learning_curves(
    eval_file=os.path.join(paths.run_dir, "evaluations.npz"),
    save_dir=paths.plots_dir,
    name_prefix=name_prefix,
    show=True,
)

## 11. Disconnect the Colab Runtime

Free the Colab GPU after the run completes so we don't keep burning compute
quota while the artifacts upload to Google Drive.


In [ ]:
import time

print("Training finished. Disconnecting runtime in 5 seconds...")
time.sleep(5)

from google.colab import runtime
runtime.unassign()